In [1]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# Load your hidden .env file for the search API key
load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

# Initialize tools required for Phase 1 Desirability market analysis
search_tool = SerperDevTool(api_key=SERPER_API_KEY)
scrape_tool = ScrapeWebsiteTool()

In [2]:

llm = LLM(
    model="bonsai-8B", 
    base_url="http://10.14.140.81:1234/v1", 
    api_key="lm-studio"                  
)

# 3. Define the Phase 1 Desirability Evaluation Agent
desirability_agent = Agent(
    role="Desirability Evaluation Agent",
    goal="Determine whether the proposed solution addresses a genuine user need and whether sufficient market demand exists.",
    backstory=(
        """You are an expert market research analyst and user experience strategist.
        Your job is to critically analyze early-stage ideas, identify real user pain points,
        assess market demand trends, and research existing competitor alternatives to find product-market fit gaps.
        Execution Requirements:
        1. Use the search tool to find 2-3 existing competitor platforms, apps, or manual alternatives addressing this problem.
        2. Scrape or analyze current market trend resources to verify if search interest or industry demand for automated coaching/AI tools is growing.
        3. Evaluate the severity of the target user pain points and assess if the features directly solve them.
        4. Synthesize all findings into a clear, cohesive report.
        """
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    max_iter=4  # Optimized to allow deep research without freezing the local machine
)

# 4. Define the Desirability Task mapped exactly to system documentation outputs
desirability_task = Task(
    description=(
        "{desirability}"
    ),
    expected_output=(
        """A formal text-formatted 'Desirability Analysis Report' containing:
        1. User Demand Analysis (validating target user pain points and problem severity).
        2. Market Demand Assessment (current industry search interest and growth indicators).
        3. Competitor Analysis (gaps, weaknesses, or friction in existing products/alternatives).
        4. Opportunity Identification (clear statement on why this solution is or is not desired by the market).
        keep the output under 800 words"""
    ),
    agent=desirability_agent
)


In [3]:
feasibility_agent = Agent(
        role="Feasibility Evaluation Agent",
        goal="Evaluate the feasibility of a startup idea strictly from the Feasibility dimension of the DFV framework.",
        backstory=(
            """You are an expert technical architect, systems analyst, and execution strategist. 
            Your task is to assess whether a startup idea can realistically be built and operated. 
            You only evaluate the Feasibility part of DFV.
            Do not evaluate desirability or viability.
            Do not give ratings, scores, grades, or percentages.
            If the idea has technical or operational weaknesses, clearly explain them and suggest improvements."""
            """Your job is to evaluate this idea strictly from the FEASIBILITY perspective of the DFV framework.
            Focus only on:
            1. Whether the product can realistically be built with current technology.
            2. What tech stack, tools, models, APIs, or infrastructure may be required.
            3. What technical and operational challenges may arise.
            4. Whether the scope is too broad, unrealistic, or difficult for a student/startup team.
            5. What changes or simplifications would make the idea more feasible.
            Important constraints:
            - Do NOT evaluate desirability.
            - Do NOT evaluate viability.
            - Do NOT give a score, rating, grade, percentage, or rank.
            - If the idea is weak, provide constructive suggestions.
            - If the idea is feasible, explain why and suggest next implementation steps."""
        ),
        llm=llm,
        tools=[search_tool, scrape_tool],
        verbose=True,
        max_iter=4
    )

feasibility_task = Task(
        description=(
                "{feasibility}"
        ),
        expected_output=(
            "A plain-language Feasibility Evaluation containing:\n"
            "1. A short feasibility opinion.\n"
            "2. Main technical and operational challenges.\n"
            "3. Required tools, stack, or infrastructure.\n"
            "4. Suggestions to improve or simplify the idea if needed.\n"
            "5. Practical next steps for implementation.\n"
            "Do not include any score, rating, grade, or percentage." \
            "keep the output under 800 words"
        ),
        agent=feasibility_agent
    )

In [4]:
viability_agent = Agent(
    role="Viability Evaluation Agent",
    goal="Determine whether the proposed solution can generate sustainable business value and long-term growth.",
    backstory=(
        """You are an expert startup strategist, business consultant, and commercialization analyst.
        Your responsibility is to evaluate business models, revenue opportunities, customer segments, 
        cost considerations, market sustainability, and long-term commercial success.

        Execution Requirements:
        1. Identify potential paying customer segments.
        2. Identify suitable business models.
        3. Analyze possible revenue streams.
        4. Assess market size and growth potential.
        5. Evaluate cost considerations.
        6. Evaluate long-term sustainability.
        7. Provide a final viability conclusion."""
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    max_iter=4
)

viability_task = Task(
    description=( "{viability}"

    ),

    expected_output=(
        "A Viability Analysis Report containing:\n"
        "1. Business Model Analysis\n"
        "2. Revenue Opportunities\n"
        "3. Customer Segment Analysis\n"
        "4. Cost Considerations\n"
        "5. Sustainability Assessment\n"
        "6. Final Viability Conclusion"
        "keep the output under 800 words"
    ),

    agent=viability_agent
)

In [5]:
dfv_risk_decision_agent = Agent(
    role="Internal DFV Decision and Risk Assessment Engine",
    goal="Identify hidden risks across project dimensions and aggregate all findings into a final project readiness decision.",
    backstory=(
        """You are an expert venture risk analyst and product strategist. Your superpower is looking
        at a product's user demand, technical stack, and business model, and instantly identifying 
        where things could go wrong (e.g., API limits, low adoption, or high maintenance costs).
        You take a supportive, coaching approach: if you find critical risks, you don't just stop the project 
        you provide a highly positive, encouraging roadmap on how to mitigate those risks and improve the idea.
        ### Required Actions (Only if NO-GO)
        [Provide a highly positive, encouraging, and constructive bulleted list of changes, 
        pivots, or mitigation strategies the team can implement to fix the gaps and safely proceed. 
        If the decision is GO, state 'No major adjustments needed; proceed with identified mitigations.']
        **CRITICAL RULE:** Do not include or expose any internal numerical scoring or percentages.
        """
    ),
    verbose=True,
    llm=llm
)

dfv_decision_task = Task(
    description=(
        """Review the reports provided in your context thoroughly from the Desirability, 
        Feasibility, and Viability evaluation phases.
        
        STEP 1: Perform a Risk Assessment based on those reports. Identify potential:
        - Technical Risks (e.g., API constraints, hallucinations, scalability issues)
        - Business Risks (e.g., market competition, adoption barriers)
        - Operational Risks (e.g., infrastructure or maintenance overhead costs)
        
        STEP 2: Synthesize the risks with the core DFV dimensions and determine if the 
        overall project is a 'GO' (Proceed) or 'NO-GO' (Needs Improvement). """
    ),
    expected_output=(
        """A structured markdown report using the following exact format:
        ## Final Decision: [GO / NO-GO]

        ### Executive Summary
        [A concise evaluation summary of the overall project strength and readiness.]
        
        ### Internal Risk Assessment Summary
        * **Technical Risks Identified:** [Brief takeaway or 'None identified']
        * **Business/Operational Risks Identified:** [Brief takeaway or 'None identified']

        ### Dimension Breakdown
        * **Desirability Summary:** [Brief takeaway from the context report]
        * **Feasibility Summary:** [Brief takeaway from the context report]
        * **Viability Summary:** [Brief takeaway from the context report]
        
        output should be in less than 12 points in case of NOGO and in case of GO a very short summary under 200 words"""
    ),
    context=[desirability_task, feasibility_task, viability_task],
    agent=dfv_risk_decision_agent
)

In [6]:
# 5. Assemble the Phase 1 Crew
crew = Crew(
    agents=[desirability_agent, feasibility_agent, viability_agent, dfv_risk_decision_agent],
    tasks=[desirability_task, feasibility_task, viability_task, dfv_decision_task],
    process=Process.sequential,
    verbose=True
)

result = await crew.kickoff_async(inputs={
    "desirability":""" Analyze the following student project proposal:
        - Customer Problem: Urban consumers need immediate access to groceries and essentials without spending time traveling to stores
        - Target Audience: Millennials, Gen Z, busy professionals, and students in metro cities aged 18-40
        - Key Value Proposition: 10-minute delivery guarantee, real-time order tracking, wide product selection
        - User Pain Points Solved: Time savings, convenience for impulse purchases, avoids crowded stores
        - Market Demand Indicators: High adoption rate in urban India, 4.2+ app rating, repeat usage frequency
        - Emotional Drivers: Convenience, instant gratification, time flexibility
                                          """, 
                                          
                                          
                                          
                                          
        "feasibility":""" The following is a startup/project idea submitted by a user:
            - Technology Stack: React Native mobile apps, cloud infrastructure, inventory management systems, route optimization algorithms
            - Infrastructure Model: Dark stores (micro-warehouses) located 2-3km from target customers, 500+ sq ft each
            - Logistics: Proprietary delivery fleet of 5,000+ delivery partners with GPS tracking
            - Supply Chain: Partnerships with 10,000+ local retailers and wholesale distributors
            - Operational Capabilities: Real-time demand forecasting, automated inventory replenishment, dynamic routing
            - Technical Challenges: Inventory accuracy, delivery time optimization, peak-hour scalability, cold chain for perishables
            - Resource Requirements: Capital investment for dark store network, technology development team, delivery workforce""", 
                                          
                                          
                                          
                                          
      "viability":""" 
        Analyze the business viability of the following project proposal:
        - Revenue Model: 
          * Delivery fees (₹29-₹59 per order)
          * Platform commissions from sellers (15-25%)
          * Advertising fees from brands
          * Blinkit Prime membership (₹99/month)
        
        - Cost Structure:
          * Dark store operational costs (rent, staffing, inventory)
          * Delivery partner payments (₹40-₹60 per delivery)
          * Technology and infrastructure costs
          * Marketing and customer acquisition costs
        
        - Market Size: India quick commerce market = $3B in 2024, projected $5-7B by 2025
        - Unit Economics: Average order value ₹300-₹500, order frequency 2-3 times/week per customer
        - Competitive Position: Market leader in 10-minute delivery, competes with Swiggy Instamart, Zepto, BigBasket
        - Profitability Status: Contribution margin positive as of 2024 (reported by Zomato)
        - Growth Trajectory: 300+ cities, 50M+ active users, 20% monthly growth
        """})

print("\n--- FINAL DESIRABILITY ANALYSIS REPORT ---\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a47f016b-84a7-4e98-9949-4896a938aadb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:  Analyze the following student project proposal:                                                         │
│          - Customer Problem: Urban consumers need immediate access to groceries and essentials without          │
│  spending time traveling to stores                                                                              │
│          - Target Audience: Millennials, Gen Z, busy professionals, and students in metro cities aged 18-40     │
│          - Key Value Proposition: 10-minute delivery guarantee, real-time order tracking, wide product          │
│  selection                                                                                                      │
│          - User Pain Points Solved: Time savings, convenience for impulse purchases, avoids crowded stores      │
│          - Market Demand Indicators: High adoption rate in urban India, 4.2+ app rating, repeat usage           │
│  frequency                                                                                                      │
│          - Emotional Drivers: Convenience, instant gratification, time flexibility                              │
│                                                                                                                 │
│  ID: 86737ee0-a766-4d83-8d0d-ff28161f167b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Desirability Evaluation Agent                                                                           │
│                                                                                                                 │
│  Task:  Analyze the following student project proposal:                                                         │
│          - Customer Problem: Urban consumers need immediate access to groceries and essentials without          │
│  spending time traveling to stores                                                                              │
│          - Target Audience: Millennials, Gen Z, busy professionals, and students in metro cities aged 18-40     │
│          - Key Value Proposition: 10-minute delivery guarantee, real-time order tracking, wide product          │
│  selection                                                                                                      │
│          - User Pain Points Solved: Time savings, convenience for impulse purchases, avoids crowded stores      │
│          - Market Demand Indicators: High adoption rate in urban India, 4.2+ app rating, repeat usage           │
│  frequency                                                                                                      │
│          - Emotional Drivers: Convenience, instant gratification, time flexibility                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'Automated grocery delivery services in metro cities market demand and competitor       │
│  analysis'}                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'Automated grocery delivery services in metro cities market demand and competitor analysis', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Online ...

...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Review the reports provided in your context thoroughly from the Desirability,                            │
│          Feasibility, and Viability evaluation phases.                                                          │
│                                                                                                                 │
│          STEP 1: Perform a Risk Assessment based on those reports. Identify potential:                          │
│          - Technical Risks (e.g., API constraints, hallucinations, scalability issues)                          │
│          - Business Risks (e.g., market competition, adoption barriers)                                         │
│          - Operational Risks (e.g., infrastructure or maintenance overhead costs)                               │
│                                                                                                                 │
│          STEP 2: Synthesize the risks with the core DFV dimensions and determine if the                         │
│          overall project is a 'GO' (Proceed) or 'NO-GO' (Needs Improvement).                                    │
│  Agent: Internal DFV Decision and Risk Assessment Engine                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: a47f016b-84a7-4e98-9949-4896a938aadb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

BadRequestError: Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4935>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}